In [7]:

%pip install qiskit qiskit-aer


In [8]:

import qiskit
import qiskit_aer

print("Qiskit version:", qiskit.__version__)
print("Qiskit Aer version:", qiskit_aer.__version__)
print("Environment ready!")


Qiskit version: 2.6.0
Qiskit Aer version: 0.17.2
Environment ready!


In [10]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Create a one-qubit circuit
qc = QuantumCircuit(1)

# Apply the Hadamard gate
qc.h(0)

# Display the circuit
print(qc.draw())

# Calculate the quantum state
state = Statevector.from_instruction(qc)

print("Statevector:", state.data)
print("Probabilities:", state.probabilities_dict())


   ┌───┐
q: ┤ H ├
   └───┘
Statevector: [0.70710678+0.j 0.70710678+0.j]
Probabilities: {np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}


In [11]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Experiment A: H followed by H
qc_a = QuantumCircuit(1)
qc_a.h(0)
qc_a.h(0)

# Experiment B: H, then Z, then H
qc_b = QuantumCircuit(1)
qc_b.h(0)
qc_b.z(0)
qc_b.h(0)

state_a = Statevector.from_instruction(qc_a)
state_b = Statevector.from_instruction(qc_b)

print("Experiment A:", state_a.probabilities_dict())
print("Experiment B:", state_b.probabilities_dict())


Experiment A: {np.str_('0'): np.float64(0.9999999999999996)}
Experiment B: {np.str_('1'): np.float64(0.9999999999999996)}


In [12]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

qc = QuantumCircuit(2)

# Prepare the state |11>
qc.x(0)
qc.x(1)

# Apply a controlled phase of pi/2
qc.cp(np.pi / 2, 0, 1)

print(qc.draw())

state = Statevector.from_instruction(qc)
print("Statevector:", state.data)


     ┌───┐         
q_0: ┤ X ├─■───────
     ├───┤ │P(π/2) 
q_1: ┤ X ├─■───────
     └───┘         
Statevector: [0.000000e+00+0.j 0.000000e+00+0.j 0.000000e+00+0.j 6.123234e-17+1.j]


In [14]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
import numpy as np

def qft_2qubit():
    qc = QuantumCircuit(2)

    qc.h(1)
    qc.cp(np.pi / 2, 1, 0)
    qc.h(0)
    qc.swap(0, 1)

    return qc

qft = qft_2qubit()

# Compare the circuit's unitary with the mathematical QFT
actual = Operator(qft).data

n = 4
j = np.arange(n)
k = np.arange(n)

expected = np.exp(2j * np.pi * np.outer(j, k) / n) / np.sqrt(n)

print(qft.draw())
print("Matches mathematical QFT:", np.allclose(actual, expected))


                   ┌───┐   
q_0: ──────■───────┤ H ├─X─
     ┌───┐ │P(π/2) └───┘ │ 
q_1: ┤ H ├─■─────────────X─
     └───┘                 
Matches mathematical QFT: True


In [15]:

for basis_index in range(4):
    initial = Statevector.from_int(basis_index, dims=(2, 2))
    final = initial.evolve(qft)

    print(f"Input {basis_index:02b}:")
    print(np.round(final.data, 3))


Input 00:
[0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j]
Input 01:
[ 0.5+0.j   0. +0.5j -0.5+0.j  -0. -0.5j]
Input 10:
[ 0.5+0.j -0.5+0.j  0.5+0.j -0.5+0.j]
Input 11:
[ 0.5+0.j  -0. -0.5j -0.5+0.j   0. +0.5j]


In [16]:

from qiskit.quantum_info import Operator, Statevector
import numpy as np

# Construct the inverse QFT
iqft = qft.inverse()

print("Inverse QFT circuit:")
print(iqft.draw())

# Verify that the two transformations cancel
combined = Operator(iqft).data @ Operator(qft).data
identity = np.eye(4)

print(
    "QFT followed by inverse QFT returns identity:",
    np.allclose(combined, identity)
)

# Test recovery of every computational basis state
for basis_index in range(4):
    initial = Statevector.from_int(basis_index, dims=(2, 2))
    recovered = initial.evolve(qft).evolve(iqft)

    print(
        f"Input {basis_index:02b} recovered:",
        np.allclose(recovered.data, initial.data)
    )


Inverse QFT circuit:
        ┌───┐               
q_0: ─X─┤ H ├─■─────────────
      │ └───┘ │P(-π/2) ┌───┐
q_1: ─X───────■────────┤ H ├
                       └───┘
QFT followed by inverse QFT returns identity: True
Input 00 recovered: True
Input 01 recovered: True
Input 10 recovered: True
Input 11 recovered: True


In [17]:

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import QFT
import numpy as np

# Two counting qubits + one eigenstate qubit
qc = QuantumCircuit(3, 2)

# Prepare the eigenstate |1>
qc.x(2)

# Put counting qubits into superposition
qc.h(0)
qc.h(1)

# Apply controlled-U operations
# U = P(pi/2), so its eigenphase on |1> is 1/4
qc.cp(np.pi / 2, 0, 2)
qc.cp(np.pi / 2, 1, 2)

# Apply inverse QFT to the counting register
# Qiskit uses little-endian qubit ordering
qc.swap(0, 1)
qc.h(1)
qc.cp(-np.pi / 2, 1, 0)
qc.h(0)

# Measure the counting qubits
qc.measure([0, 1], [0, 1])

print(qc.draw())


     ┌───┐                                    ┌───┐┌─┐
q_0: ┤ H ├─■─────────────────X───────■────────┤ H ├┤M├
     ├───┤ │                 │ ┌───┐ │P(-π/2) └┬─┬┘└╥┘
q_1: ┤ H ├─┼────────■────────X─┤ H ├─■─────────┤M├──╫─
     ├───┤ │P(π/2)  │P(π/2)    └───┘           └╥┘  ║ 
q_2: ┤ X ├─■────────■───────────────────────────╫───╫─
     └───┘                                      ║   ║ 
c: 2/═══════════════════════════════════════════╩═══╩═
                                                1   0 


In [18]:

from qiskit_aer import AerSimulator

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()

counts = result.get_counts()
print("QPE measurement counts:", counts)


QPE measurement counts: {'10': 479, '00': 284, '01': 261}


In [20]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.synthesis import synth_qft_full
import numpy as np

qc = QuantumCircuit(3, 2)

# Prepare the eigenstate |1>
qc.x(2)

# Prepare the two counting qubits
qc.h(0)
qc.h(1)

# Controlled-U and controlled-U^2
qc.cp(np.pi / 2, 0, 2)
qc.cp(np.pi, 1, 2)

# Build the inverse QFT using elementary gates
iqft = synth_qft_full(2, inverse=True, do_swaps=True)
qc.compose(iqft, qubits=[0, 1], inplace=True)

# Measure the counting register
qc.measure([0, 1], [0, 1])

print(qc.draw())

# Transpile for Aer compatibility
simulator = AerSimulator()
compiled = transpile(qc, simulator, optimization_level=1)

result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print("QPE measurement counts:", counts)


     ┌───┐                   ┌───┐               ┌─┐   
q_0: ┤ H ├─■───────────────X─┤ H ├─■─────────────┤M├───
     ├───┤ │               │ └───┘ │P(-π/2) ┌───┐└╥┘┌─┐
q_1: ┤ H ├─┼────────■──────X───────■────────┤ H ├─╫─┤M├
     ├───┤ │P(π/2)  │P(π)                   └───┘ ║ └╥┘
q_2: ┤ X ├─■────────■─────────────────────────────╫──╫─
     └───┘                                        ║  ║ 
c: 2/═════════════════════════════════════════════╩══╩═
                                                  0  1 
QPE measurement counts: {'01': 1024}
